# **TRABALHO DA UNIDADE 1 - ALGORITMO E ESTRUTURA DE DADOS 2**
**Docente:** Ivanovitch Silva  
**Discentes:** Felipe Gabriel Bezerra da Silva e Laíze Suélia da Silva Oliveira  

Este notebook tem como objetivo processar documentos não estruturados (PDFs), extrair Entidades Nomeadas (NER) utilizando Processamento de Linguagem Natural (NLP) e construir um grafo de co-ocorrência. O resultado final mapeia as relações e identifica os principais *hubs* (conectores) num contexto de análise de risco financeiro.


- Construir pipeline de análise de dados capaz de ler documentos nao estruturados;

- Relatórios financeiros e jurídicos do caso do Banco Master


#

## 1. Configuração do Ambiente e Bibliotecas

Instalação do spaCy e carregamento dos PDFs do Drive.

Nesta etapa, preparamos o ambiente instalando as bibliotecas essenciais para a pipeline:

- extração de PDF (`PyMuPDF`)
- processamento de linguagem (`spaCy`)
- matemática e visualização de redes complexas (`NetworkX` e `PyVis`).

In [ ]:
!pip install pymupdf spacy networkx
!python -m spacy download pt_core_news_lg  #Modelo de português do spaCy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 38.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 2.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
from google.colab import drive
import fitz  # PyMuPDF
import spacy
import re
import itertools

drive.mount('/content/drive')

nlp = spacy.load("pt_core_news_lg")

caminhos = [
    "/content/drive/MyDrive/PROJETO1/arquivo1_banco_master.pdf",
    "/content/drive/MyDrive/PROJETO1/arquivo2_banco_master.pdf",
    "/content/drive/MyDrive/PROJETO1/arquivo3_banco_master.pdf"
]

Mounted at /content/drive


# Extração das entidades (NER) com LIMPEZA

Modelos generalistas de NLP costumam cometer erros de classificação.
Para garantir a fidelidade dos dados, definimos:

* **Termos Proibidos:** Ruído comum em documentos oficiais (como "Fls.", "TC-", "CÓPIA") que deve ser ignorado.
* **Normalização:** Uma regra de força bruta para garantir que os termos derivados sejam agrupados na mesma entidade.

In [ ]:
termos_proibidos = [
    'https://', 'http', 'www.', 'TC-', 'Fls.', 'Fls', 'CNPJ', 'SEI',
    'Documento Digital', 'Validar', 'CÓPIA', 'Original', 'Doc', 'Anexo',
    'assinatura', 'e-TCESP', 'Processo', 'inciso', 'art.', 'Lei',
    'Resolução', 'Parágrafo', 'Portaria', 'Constituição',
    'QUANTO POR OMISSÃO', 'OMISSÃO', 'RELATÓRIO', 'LF)12', 'LRF64',
    'CF18', 'CF21', 'CF28', 'SP22', 'SP67', 'II.2', 'II)27',
    'VI)24', 'VI)25', 'DAIR59', 'CVM31', 'RIRPP5','30/12/2025 46 Disponível', 'Administração19',
    'II.3', 'II.3.2 Do', 'II.3.3', 'Índice', 'Alta Tensão”17', 'Alta Tensão”27', 'Atestado', 'Nesta%20sexta%2Dfeira%20(3),e%20garantias%20co...',
    'C.2 Gestão dos Investimentos','CF','CARE11','DOS PREJUÍZOS HISTÓRICOS DA CARTEIRA DO RPPS DE', 'Disponível'
]

# EXTRAÇÃO POR PARÁGRAFOS (BLOCOS)
paragrafos_arquivos = {}

for caminho in caminhos:
    nome_arquivo = caminho.split('/')[-1]
    print(f"A extrair blocos de: {nome_arquivo}...")
    doc = fitz.open(caminho)
    paragrafos_limpos = []

    for pagina in doc:
        blocos = pagina.get_text("blocks")
        for bloco in blocos:
            texto_bloco = bloco[4]
            p_limpo = texto_bloco.replace('\n', ' ')
            p_limpo = re.sub(r'\s+', ' ', p_limpo).strip()

            if len(p_limpo) < 15: continue

            if "C Ó P IA D E D O C U M E N T O" in p_limpo: continue
            if "MINISTÉRIO PÚBLICO DE CONTAS" in p_limpo: continue
            if "Avenida Rangel Pestana" in p_limpo or "mpc.sp.gov.br" in p_limpo: continue
            if "original acesse http://e-processo.tce.sp.gov.br" in p_limpo: continue

            p_limpo = re.sub(r'TC-\d+\.\d+\.\d+-\d+', '', p_limpo)
            p_limpo = re.sub(r'Fls\.\s*\d+', '', p_limpo)
            p_limpo = re.sub(r'[A-Z0-9-]{10,}', '', p_limpo)
            p_limpo = re.sub(r'\s+', ' ', p_limpo).strip()

            if len(p_limpo) > 10: paragrafos_limpos.append(p_limpo)

    paragrafos_arquivos[nome_arquivo] = paragrafos_limpos

# PROCESSAMENTO NLP (NER) NAS ENTIDADES
resultados_por_arquivo = {}
arestas_projeto = []

for nome_arquivo, paragrafos in paragrafos_arquivos.items():
    print(f"A processar NLP para: {nome_arquivo}...")
    resultados_por_arquivo[nome_arquivo] = {"PESSOAS": [], "ORGS": [], "LOCS": [], "DIVERSOS": []}
    entidades_do_arquivo_todas = []

    for p_limpo in paragrafos:
        doc_p = nlp(p_limpo)
        entidades_do_paragrafo = []

        for ent in doc_p.ents:
            termo = ent.text.strip()

            # Normalização
            if "Banco Master" in termo: termo = "Banco Master"
            if termo in ["Bluemac Asset Management", "Bluemac Asset Management LTDA.", "Macam Asset", "Master Capital Asset", "Bluemac Asset"]: termo = "Bluemac Asset"
            if termo in ["Texas I Fundo de Investimento em Ações", "Texas I FI Ações", "Fundo Texas I FI Ações", "Texas"]: termo = "Fundo Texas"
            if termo in ["Nest Eagle FII", "Next [sic] Eagle", "EAGL11", "Fundo Imobiliário Next [sic] Eagle"]: termo = "FII Nest Eagle"
            if termo in ["Vamos Investir Brasil", "Vamos Investir Brasil – Assessor de Investimentos Ltda", "Vibra Investimentos17"]: termo = "Vibra Investimentos"
            if termo in ["Tribunal de Contas do Estado de São Paulo", "Tribunal de Contas do Estado", "TCE-SP", "Tribunal de Contas da"]: termo = "Tribunal de Contas"
            if termo in ["o presidente", "o Presidente", "o presidente do"]: termo = "Presidente"
            if termo in ["o Senador", "o Senador do"]: termo = "Senador"
            if termo in ["10 Descrição de Segmentos", "Descrição de Segmentos"] : termo = "Descrição de Segmentos"
            if termo in ["Comitê de Investimentos de 17/10/2025", "Comitê de Investimentos do RPPS", "Comitê de Investimentos18"]: termo = "Comitê de Investimentos"
            if termo in ["Banco Central", "Banco Central do Brasil"]: termo = "Banco Central do Brasil"
            if termo in ["Banco Master", "Banco Master S/A."]: termo = "Banco Master"
            if termo in ["Demonstrativo de Aplicações", "Demonstrativo de Aplicações e Investimentos do...", "DAIR"]: termo = "Demonstrativo de Aplicações e Investimentos do Rercursos"
            if termo in ["Ministério da Previdência18"]: termo = "Ministério da Previdência"
            if termo in ["IPCA+7,10"]: termo = "IPCA"
            if termo in ["Macan A7"]: termo = "Macan"



            termo = termo.replace("BRODOWSKI", "Brodowski")
            termo = termo.replace("BRASIL", "Brasil").replace("Brasil39", "Brasil")
            termo = termo.replace("SP", "São Paulo").replace("Estado de São Paulo", "São Paulo").replace("de São Paulo", "São Paulo").replace( "SISão PauloREV", "São Paulo")
            termo = termo.replace("Câmara Municipal de Brodowski", "Câmara Municipal").replace("Câmara de Vereadores de Brodowski", "Câmara Municipal").replace("Câmara Municipal10", "Câmara Municipal")
            termo = termo.replace("Sr. Flávio Araújo Guidolin Silva", "Flávio Araújo Guidolin Silva")
            termo = termo.replace('500 milhões8', '500 milhões')
            termo = termo.replace("AL Maceió", "Maceió")
            termo = termo.replace("RPPS", "Regime Próprio de Previdência Social")
            termo = termo.replace("Banco Master S/A.", "Banco Master").replace("Banco Master S/A", "Banco Master")

            if ent.label_ in ["PER", "ORG", "LOC", "MISC"]:
                if not any(proibido.lower() in termo.lower() for proibido in termos_proibidos):
                    if len(termo) > 3:
                        #categoria_final = correcao_categorias.get(termo, ent.label_)
                        entidades_do_paragrafo.append(termo)
                        if ent.label_ == "PER": resultados_por_arquivo[nome_arquivo]["PESSOAS"].append(termo)
                        elif ent.label_ == "ORG": resultados_por_arquivo[nome_arquivo]["ORGS"].append(termo)
                        elif ent.label_ == "LOC": resultados_por_arquivo[nome_arquivo]["LOCS"].append(termo)
                        elif ent.label_ == "MISC": resultados_por_arquivo[nome_arquivo]["DIVERSOS"].append(termo)

        # Cria arestas de co-ocorrência
        entidades_do_paragrafo = list(set(entidades_do_paragrafo))
        if len(entidades_do_paragrafo) > 1:
            for u, v in itertools.combinations(entidades_do_paragrafo, 2):

                arestas_projeto.append((u, v))

        #Guarda as entidades para a contagem final
        entidades_do_arquivo_todas.extend(entidades_do_paragrafo)

    qtd_unicas = len(set(entidades_do_arquivo_todas))
    print(f"-> {qtd_unicas} entidades únicas registadas em {nome_arquivo}")

print("Extração e Processamento NLP concluídos com sucesso!")

A extrair blocos de: arquivo1_banco_master.pdf...
A extrair blocos de: arquivo2_banco_master.pdf...
A extrair blocos de: arquivo3_banco_master.pdf...
A processar NLP para: arquivo1_banco_master.pdf...
-> 83 entidades únicas registadas em arquivo1_banco_master.pdf
A processar NLP para: arquivo2_banco_master.pdf...
-> 118 entidades únicas registadas em arquivo2_banco_master.pdf
A processar NLP para: arquivo3_banco_master.pdf...
-> 185 entidades únicas registadas em arquivo3_banco_master.pdf
Extração e Processamento NLP concluídos com sucesso!


# Geração do grafo


In [ ]:
!pip install plotly -q

In [ ]:
import networkx as nx
import plotly.graph_objects as go

# 1. CRIAR O GRAFO E ISOLAR O NÚCLEO PRINCIPAL
G = nx.Graph()
for u, v in arestas_projeto:
    if G.has_edge(u, v): G[u][v]['weight'] += 1
    else: G.add_edge(u, v, weight=1)

maior_componente = max(nx.connected_components(G), key=len)
G_main = G.subgraph(maior_componente).copy()

# 2. MAPEAR CORES E CATEGORIAS
entidade_para_label = {}
for arquivo, categorias in resultados_por_arquivo.items():
    for ent in categorias["PESSOAS"]: entidade_para_label[ent] = 'PER'
    for ent in categorias["ORGS"]: entidade_para_label[ent] = 'ORG'
    for ent in categorias["LOCS"]: entidade_para_label[ent] = 'LOC'
    for ent in categorias["DIVERSOS"]: entidade_para_label[ent] = 'MISC'

cores_dict = {
    'PER': '#00E5FF',  # Ciano Neon
    'ORG': '#FFEA00',  # Amarelo Neon
    'LOC': '#00E676',  # Verde Neon
    'MISC': '#D500F9'  # Magenta Neon
}

# 3. CALCULAR O LAYOUT
# Usar weight=None impede que os nós sejam esmagados no centro
# Aumentar o k e o número de iterações espalha mais a rede
pos = nx.spring_layout(G_main, k=0.9, iterations=400, weight=None, seed=42)
graus = dict(G_main.degree())

top_10_hubs = sorted(graus, key=graus.get, reverse=True)[:10]

# 4. PREPARAR AS LINHAS (Arestas)
edge_x = []
edge_y = []
for edge in G_main.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=0.4, color='rgba(255, 255, 255, 0.2)'), # Linhas mais finas e subtis
    hoverinfo='none',
    mode='lines')

# 5. PREPARAR OS PONTOS E TEXTOS (Nós)
node_x = []
node_y = []
node_hover_text = []
node_static_text = []
node_color = []
node_size = []

for node in G_main.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)

    categoria = entidade_para_label.get(node, 'MISC')
    grau = graus[node]

    node_hover_text.append(f"<b>{node}</b><br>Categoria: {categoria}<br>Conexões: {grau}")
    node_color.append(cores_dict[categoria])

    # Nós significativamente menores para não poluir
    node_size.append(6 + (grau * 1.0))

    if node in top_10_hubs:
        node_static_text.append(f"<b>{node}</b>")
    else:
        node_static_text.append("")

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    hoverinfo='text',
    hovertext=node_hover_text,
    text=node_static_text,
    textposition="top center",
    textfont=dict(color="white", size=10), # Fonte ligeiramente menor
    marker=dict(
        showscale=False,
        color=node_color,
        size=node_size,
        line_width=1,
        line_color='white'
    ))

# 6. MONTAR A FIGURA FINAL
fig = go.Figure(data=[edge_trace, node_trace],
             layout=go.Layout(
                height=800, # Aumenta a área vertical para espalhar os nós
                title=dict(text="Grafo de Co-ocorrência em Parágrafos (Top 10 Hubs)", font=dict(size=22, color="white")),
                titlefont_size=16,
                showlegend=False,
                hovermode='closest',
                margin=dict(b=0, l=0, r=0, t=50), # Remove as margens pretas em excesso
                plot_bgcolor='#0D1117',
                paper_bgcolor='#0D1117',
                xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor="x", scaleratio=1))
                )

# Adiciona a legenda
legendas = [
    dict(xref="paper", yref="paper", x=0.01, y=0.98, text="Pessoas Físicas", showarrow=False, font=dict(color="#00E5FF", size=14)),
    dict(xref="paper", yref="paper", x=0.01, y=0.95, text="Bancos/Organizações", showarrow=False, font=dict(color="#FFEA00", size=14)),
    dict(xref="paper", yref="paper", x=0.01, y=0.92, text="Localidades", showarrow=False, font=dict(color="#00E676", size=14)),
    dict(xref="paper", yref="paper", x=0.01, y=0.89, text="Diversos (MISC)", showarrow=False, font=dict(color="#D500F9", size=14))
]

fig.update_layout(annotations=legendas)
fig.show()

# Conferência Geral

In [9]:
#import pandas as pd

#conferencia_geral = []
#for arquivo, categorias in resultados_por_arquivo.items():
#    for label, lista in categorias.items():
#        for item in set(lista):
#            if item in G.nodes():
#                conferencia_geral.append({
#                    "Entidade": item,
#                    "Label_IA": label,
#                    "Arquivo": arquivo,
#                    "Grau": G.degree(item)
#                })

#df_conferir = pd.DataFrame(conferencia_geral).sort_values(by="Grau", ascending=False)
#pd.set_option('display.max_rows', None)
#display(df_conferir)

# Análise dos resultados
#### Como forma de analisar a rede, foram extraídas algumas métricas do Grafo completo (G) e feita a comparação com os valores referentes ao nó principal (G_main).

- **Nó principal:** diz respeito ao maior caminho do grafo.
- **Número de nós:** corresponde à quantidade de elementos no gráfico.
- **Número de arestas:** corresponde à quantidade de conexões (correlações) entre os elementos.
- **Grau médio:** informa média dos graus (quantidade de conexões) de cada elemento.
- **Densidade:** representa a razão entre o número de arestas do grafo e o número máximo de conexões.

## Conclusões -
1. O nó principal representa mais de **65%** da rede completa.
2. As arestas do nó principal representam mais **85%** das arestas da rede completa.
3. A densidade do nó principal é maior que a densidade da rede completa, indicando que nesse nó existem mais entidades em correlação.
4. Aproximadamente **35%** das entidades possuem pouca ou nenhuma correlação com o nó principal.

In [ ]:
valor_dos_graus = [grau for _, grau in G.degree()] # Cria uma lista com o grau dos nós do grafo
media_graus = sum(valor_dos_graus) / len(valor_dos_graus) # Calcula o valor médio dos graus

valor_dos_graus_main = [grau for _, grau in G_main.degree()] # Cria uma lista com o grau dos nós do grafo
media_graus_main = sum(valor_dos_graus_main) / len(valor_dos_graus_main) # Calcula o valor médio dos graus

#Criando um grafo com o nó principal do grafo completo
grau = dict(G_main.degree())

metricas = {
    'Métrica': ['Número de nós:','Número de arestas:','Grau médio:','Densidade:'],
    'Rede Completa': [G.number_of_nodes(),
                      G.number_of_edges(),
                      media_graus,
                      nx.density(G)],
    'Nó principal': [G_main.number_of_nodes(),
                      G_main.number_of_edges(),
                      media_graus_main,
                      nx.density(G_main)],
    'Porcentagem': [(G_main.number_of_nodes())/(G.number_of_nodes()),
                    (G_main.number_of_edges())/(G.number_of_edges()),
                    '-','-']
}
df_metricas = pd.DataFrame(metricas)
df_metricas_style = (
    df_metricas.style
    .format({ 'Rede Completa': lambda x: f'{int(x)}' if float(x).is_integer() else f'{x:.3f}', #Mostra int inteiro e float com decimal
              'Nó principal': lambda x: f'{int(x)}' if float(x).is_integer() else f'{x:.3f}', #Mostra int inteiro e float com decimal
              'Porcentagem': lambda x: f'{x*100:.2f}%' if isinstance(x, (int, float)) else x
})
    .hide(axis='index')
    .set_table_styles([
        # Cabeçalho centralizado e em negrito
        {'selector': 'th',
         'props': [
             ('text-align', 'center'),
             ('font-weight', 'bold')
         ]},
        # Valores alinhados à esquerda
        {'selector': 'td',
         'props': [
             ('text-align', 'left')
         ]},
        # Título da tabela
        {'selector': 'caption',
        'props': [
            ('font-size', '16px'),
            ('font-weight', 'bold'),
            ('text-align', 'center'),
            ('padding', '10px')
        ]}
    ])
    .set_caption("ANÁLISE DAS MÉTRICAS DA REDE")
) #Estilização do dataframe
display(df_metricas_style)
print()

import plotly.express as px

df_plot = df_metricas[df_metricas['Métrica'].isin(['Número de nós:', 'Número de arestas:'])]
df_melted = df_plot[['Métrica', 'Rede Completa', 'Nó principal']].melt(id_vars='Métrica', var_name='Tipo de Rede', value_name='Quantidade')

fig = px.bar(df_melted,
             x='Métrica',
             y='Quantidade',
             color='Tipo de Rede',
             barmode='group',
             text='Quantidade',
             title='Comparação: Rede Completa vs. Nó Principal',
             color_discrete_map={'Rede Completa': '#00E5FF', 'Nó principal': '#FFEA00'})

fig.update_traces(
    textposition='outside' # <--- Coloca o valor acima da barra
)

# 3. Estilização Neon
fig.update_layout(
    plot_bgcolor='#0D1117',
    paper_bgcolor='#0D1117',
    font_color="white",
    bargap=0.2
)

fig.show()

Métrica,Rede Completa,Nó principal,Porcentagem
Número de nós:,192,127,66.15%
Número de arestas:,513,452,88.11%
Grau médio:,5.344,7.118,-
Densidade:,0.028,0.056,-


### Contextualizando os resultados
Com a tabela abaixo, conseguimos identificar as 10 entidades com maiores conexões.

- A métrica 'Intermediação (Ponte)' entra nesse cenário indicando um valor referente à entidade faz uma ponte para as demais entidades.
- A métrica 'Proximidade (Centro)' apresenta o termo central da rede.
</br>





In [ ]:
# Calcular diferentes tipos de importância para o núcleo principal
grau = dict(G_main.degree())
intermediacao = nx.betweenness_centrality(G_main) # Mede quem serve de "ponte"
proximidade = nx.closeness_centrality(G_main)    # Mede quem está no centro da rede

# Criar um DataFrame consolidado
df_analise = pd.DataFrame({
    'Grau (Conexões)': grau,
    'Intermediação (Ponte) ↓': intermediacao,
    'Proximidade (Centro)': proximidade
})
df_analise = df_analise.reset_index().rename(columns={'index': 'Entidade'})
df_ranking = df_analise.sort_values(by='Intermediação (Ponte) ↓', ascending=False).head(10)

df_analise_style = (
    df_ranking.style
    .hide(axis='index')
    .set_table_styles([
        # Cabeçalho centralizado e em negrito
        {'selector': 'th',
         'props': [
             ('text-align', 'center'),
             ('font-weight', 'bold')
         ]},

        # Valores alinhados à esquerda
        {'selector': 'td',
         'props': [
             ('text-align', 'left')
         ]},
        # Título da tabela
        {'selector': 'caption',
        'props': [
            ('font-size', '16px'),
            ('font-weight', 'bold'),
            ('text-align', 'center'),
            ('padding', '10px')
        ]}
    ])
    .set_caption("RANKING TOP 10 ENTIDADES MAIS CONECTADAS")
)
# Ordenar por intermediação para ver quem "controla" o fluxo entre os nós
display(df_analise_style)
print()

# Gerando o gráfico
fig_barras = px.bar(df_ranking,
                    x='Grau (Conexões)',
                    y='Entidade',
                    orientation='h',
                    title='Top 10 Entidades por Número de Conexões',
                    text='Grau (Conexões)',
                    color='Grau (Conexões)',
                    color_continuous_scale='Viridis')

# Ajustes estéticos para combinar com o fundo escuro do seu projeto
fig_barras.update_layout(
    plot_bgcolor='#0D1117',
    paper_bgcolor='#0D1117',
    font_color="white",
    yaxis={'categoryorder':'total ascending'} # Garante que o maior fique no topo
)

fig_barras.show()

Entidade,Grau (Conexões),Intermediação (Ponte) ↓,Proximidade (Centro)
Regime Próprio de Previdência Social,41,0.411589,0.577982
Fundo Texas,37,0.310298,0.540773
São Paulo,31,0.214926,0.529412
Brodowski,28,0.205432,0.525000
Banco Central do Brasil,20,0.095856,0.451613
Banco Master,12,0.076368,0.431507
Comitê de Investimentos,16,0.068707,0.411765
Demonstrativo de Aplicações e Investimentos do Rercursos,12,0.054387,0.411765
Política de Investimentos,18,0.050123,0.463235
São Roque,18,0.034565,0.382979


- Para as duas métricas acima, o termo **'Regime Próprio de Previdência Social'** aparece como a entidade com os maiores valores, ou seja, é o termo que mais faz relação com outras entidades e é o tema central nos três arquivos.
- Em segunda lugar no ranking, temos o termo **'Fundo Texas'**, com valores próximos ao primeiro lugar.

> **Em conclusão, pode-se admitir que os processos judiciais analisados pelo MPC-SP têm como eixo central a investigação de valores milionários investidos, majoritariamente, no Fundo Texas. As evidências encontradas apontam que os principais atores afetados são os Institutos de Previdência Municipal (RPPS), cujos recursos utilizados foram expostos a grandes riscos financeiros. Embora não seja o ator principal em todos os processos, um grande risco identificado foi o aporte de altas quantias no Banco Master, instituição que esteve envolvida em irregularidades financeiras e judiciais nos últimos anos.**